# 2024년 10월 17일 GTX-A 광역버스 교통카드 합성데이터 조회

33개 버스번호를 기준으로 공공데이터포털 API를 호출한다.

- `opr_ymd`: `20241017`
- `users_type_cd`: `01`
- `ride_ctpv_cd`: `41`
- `rte_id`: CSV의 버스번호(`1000`, `M7111` 등)

In [ ]:
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BASE_URL = "https://apis.data.go.kr/1613000/RegionalTransportationCardUsageSyntheticData/getGyeonggiTransportationCardUsageSyntheticData"
SERVICE_KEY = getpass("공공데이터포털 인증키를 입력하세요: ")
OPR_YMD = "20241017"
USERS_TYPE_CD = "01"
RIDE_CTPV_CD = "41"
PAGE_SIZE = 1000

print("설정 완료")

In [ ]:
# 33개 노선번호 읽기
route_file = Path("gtx_a_seoul_bus_outputs/gtx_a_north_33_unique_routes.csv")
routes = pd.read_csv(route_file, dtype=str, encoding="utf-8-sig")

# CSV에 표시용으로 붙어 있는 앞쪽 작은따옴표 제거
routes["버스번호"] = routes["버스번호"].str.strip().str.lstrip("'")
routes = routes.drop_duplicates(subset=["버스번호"]).reset_index(drop=True)

print(f"조회할 노선 수: {len(routes)}")
display(routes[["버스번호", "GTX_A_역", "대상지역", "노선ID"]])

In [ ]:
def parse_response(payload):
    # API 응답이 response로 한 번 감싸지는 경우와
    # header/body가 바로 오는 경우를 모두 처리
    response = payload.get("response", payload.get("Response", payload))
    header = response.get("header", {}) or {}
    body = response.get("body", {}) or {}
    items = body.get("items", {}) or {}
    items = items.get("item", []) if isinstance(items, dict) else items
    if isinstance(items, dict):
        items = [items]
    if not isinstance(items, list):
        items = []
    return items, header, body

def request_route(route_no, page_no=1):
    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": page_no,
        "numOfRows": PAGE_SIZE,
        "dataType": "JSON",
        "opr_ymd": OPR_YMD,
        "rte_id": route_no,
        "users_type_cd": USERS_TYPE_CD,
        "ride_ctpv_cd": RIDE_CTPV_CD,
    }
    r = requests.get(BASE_URL, params=params, timeout=60)
    print("요청 URL(인증키 제외):", r.url.split("serviceKey=")[0] + "serviceKey=***")
    print("HTTP 상태코드:", r.status_code)
    r.raise_for_status()
    return parse_response(r.json())

In [ ]:
# 먼저 1000번 노선의 STCIS 노선 ID 41016025를 1페이지 시험 조회
# 조회일자: 2024년 10월 17일 (OPR_YMD = 20241017)
test_items, test_header, test_body = request_route("41016025", page_no=1)
print("header:", test_header)
print("body 주요 키:", list(test_body.keys()))
print("조회 건수:", len(test_items))
if test_items:
    display(pd.DataFrame(test_items).head())

In [ ]:
# 2025-10-16: final 32 routes, queried by STCIS route ID
TARGET_DATE = "20251016"
OPR_YMD = TARGET_DATE
route_id_map = {
    "41016025": "1000", "41016018": "1082", "41016045": "1100",
    "41016027": "1200", "41016044": "1500", "41016055": "1500-reserved",
    "41016042": "1900", "41084003": "2200", "41084009": "2200-1",
    "41084007": "3100", "41084006": "3100N", "41016049": "3400",
    "41011156": "6701", "41016050": "7101", "41084008": "7111",
    "41016903": "M7111", "41016904": "M7111-reserved", "41084903": "M7154",
    "41003902": "M7412", "41003910": "M7412-reserved", "41084001": "G7426",
    "41165001": "7602", "41084002": "G7625", "41016902": "M7731",
    "41084004": "9030", "41084005": "9030-1", "41003717": "9600",
    "41016029": "9700", "41034153": "9709", "41034156": "9709N",
    "41034123": "9710", "41034145": "9710-1",
}

all_rows = []
results = []
for i, (route_id, route_no) in enumerate(route_id_map.items(), start=1):
    route_rows = []
    try:
        page_no = 1
        while True:
            items, header, body = request_route(route_id, page_no=page_no)
            route_rows.extend(items)
            total = int(body.get("totalCount", body.get("totalcount", 0)) or 0)
            if not items or (total and len(route_rows) >= total) or len(items) < PAGE_SIZE:
                break
            page_no += 1
        for row in route_rows:
            row["query_route_id"] = route_id
            row["query_route_no"] = route_no
            row["query_date"] = TARGET_DATE
        all_rows.extend(route_rows)
        results.append({"route_id": route_id, "route_no": route_no, "status": "success", "count": len(route_rows), "error": ""})
        print(f"[{i}/{len(route_id_map)}] {route_no} ({route_id}): {len(route_rows)} rows")
    except Exception as e:
        results.append({"route_id": route_id, "route_no": route_no, "status": "failed", "count": 0, "error": repr(e)})
        print(f"[{i}/{len(route_id_map)}] {route_no} ({route_id}): failed - {e}")
    time.sleep(0.2)

result_df = pd.DataFrame(results)
raw_df = pd.DataFrame(all_rows)
print("Total API response rows:", len(raw_df))
display(result_df)


In [ ]:
# Display-only summary. No output files are written.
if not raw_df.empty and "utztn_nope" in raw_df.columns:
    raw_df["utztn_nope"] = pd.to_numeric(raw_df["utztn_nope"], errors="coerce").fillna(0)
    summary = (raw_df.groupby(["query_route_id", "query_route_no"], as_index=False)
               .agg(usage_count=("utztn_nope", "sum"), api_response_rows=("query_route_id", "size")))
else:
    summary = result_df.copy()

display(summary)
